# Practical 04: Out-of-Core Batch Ingestion & Aggregation via Chunking

**UNIT 1: Introduction to Big Data and Large Dataset Handling**

**Problem Statement:** Write a Python program to ingest a large CSV file exceeding single-process RAM in batches using the chunksize parameter of pd.read_csv(). Calculate the cumulative record count, running sum of transaction amounts, and overall average transaction value across all chunks to prevent Out-Of-Memory (OOM) crashes.

In [1]:
import os
import shutil
import tempfile
import numpy as np
import pandas as pd

TOTAL_ROWS = 700_000
WRITE_BATCH = 140_000
CHUNK_SIZE = 70_000

rng = np.random.default_rng(123)
tmp_dir = tempfile.mkdtemp()
csv_path = os.path.join(tmp_dir, "large_transactions.csv")

written = 0
while written < TOTAL_ROWS:
    m = min(WRITE_BATCH, TOTAL_ROWS - written)
    batch = pd.DataFrame({
        "transaction_id": np.arange(written + 1, written + m + 1),
        "customer_id": rng.integers(1, 100000, m),
        "amount": np.round(rng.gamma(2.0, 45.0, m), 2),
        "city": rng.choice(["Surat", "Mumbai", "Pune", "Nagpur", "Thane"], m),
    })
    batch.to_csv(csv_path, mode="a", header=(written == 0), index=False)
    written += m

print(f"File created: {os.path.getsize(csv_path) / 1024**2:.1f} MB, {TOTAL_ROWS:,} rows")

count_all = 0
grand_sum = 0.0

stream = pd.read_csv(csv_path, usecols=["transaction_id", "amount"], chunksize=CHUNK_SIZE)

print(f"{'Chunk':>5} | {'Rows':>8} | {'Cum. records':>12} | {'Running sum':>16} | {'Running avg':>11}")
print("-" * 66)
for i, chunk in enumerate(stream, start=1):
    count_all += len(chunk)
    grand_sum += chunk["amount"].sum()
    print(f"{i:>5} | {len(chunk):>8,} | {count_all:>12,} | {grand_sum:>16,.2f} | "
          f"{grand_sum / count_all:>11.4f}")

overall_avg = grand_sum / count_all
print(f"Total records             : {count_all:,}")
print(f"Total transaction amount  : {grand_sum:,.2f}")
print(f"Average transaction value : {overall_avg:.4f}")

full = pd.read_csv(csv_path, usecols=["amount"])["amount"]
print(f"Full-load sum : {full.sum():,.2f} | mean : {full.mean():.4f}")

shutil.rmtree(tmp_dir)
print("Temporary files removed.")

File created: 16.8 MB, 700,000 rows
Chunk |     Rows | Cum. records |      Running sum | Running avg
------------------------------------------------------------------
    1 |   70,000 |       70,000 |     6,314,207.67 |     90.2030
    2 |   70,000 |      140,000 |    12,637,828.88 |     90.2702
    3 |   70,000 |      210,000 |    18,948,671.75 |     90.2318
    4 |   70,000 |      280,000 |    25,277,887.62 |     90.2782
    5 |   70,000 |      350,000 |    31,571,883.84 |     90.2054
    6 |   70,000 |      420,000 |    37,863,445.92 |     90.1511
    7 |   70,000 |      490,000 |    44,159,381.16 |     90.1212
    8 |   70,000 |      560,000 |    50,485,330.63 |     90.1524
    9 |   70,000 |      630,000 |    56,783,229.59 |     90.1321
   10 |   70,000 |      700,000 |    63,096,509.79 |     90.1379
Total records             : 700,000
Total transaction amount  : 63,096,509.79
Average transaction value : 90.1379
Full-load sum : 63,096,509.79 | mean : 90.1379
Temporary files remov